# **Configuracion y entrenamiento CPU en Colab**

Primero, se prepara el repositorio y se verifican las dependencias y la GPU. Luego,
se ejecuta un kernel propio y se cargan los tensores desde Drive. Finalmente, se
puede entrenar la CNN en CPU y respaldar los pesos y resultados en Drive.

En una sesion nueva se usa la clonacion; si la carpeta ya existe y se hizo un push,
se usa la actualizacion. Se ejecuta una de esas dos opciones y despues se continua
con las demas secciones. Si cambian modulos Python ya importados, se debe reiniciar
la sesion antes de volver a utilizarlos; Git no actualiza esos modulos en memoria.


## **Clonar el repositorio por primera vez**

Se ejecuta cuando la carpeta del proyecto todavia no existe en Colab. Se descarga
la revision seleccionada y se muestra el commit para poder identificar el codigo
utilizado. Si la carpeta ya existe, se usa la siguiente seccion.


In [ ]:
from pathlib import Path
import subprocess

url_repositorio = "https://github.com/Bamo0507/Proyecto_3_Cuda_CNN.git"
revision_repositorio = "main"
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

if directorio_proyecto.exists():
    raise RuntimeError("La carpeta ya existe. Usa la seccion de actualizacion.")

subprocess.run(
    ["git", "clone", url_repositorio, str(directorio_proyecto)],
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", revision_repositorio],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Actualizar el repositorio despues de un push**

Se ejecuta si la carpeta ya existe y se quieren descargar los cambios publicados
en `main`, sin volver a clonar. Esto actualiza los archivos del proyecto, pero no
las celdas del notebook abierto ni los modulos Python ya importados. No se necesita
esta opcion inmediatamente despues de clonar la misma version.


In [ ]:
from pathlib import Path
import subprocess

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=directorio_proyecto,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", "origin/main"],
    cwd=directorio_proyecto,
    check=True,
)
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Commit ejecutado:", commit_ejecutado)


## **Instalar las dependencias de Colab**

Se ejecuta al preparar el entorno o cuando cambian las dependencias. Se utiliza
el archivo del repositorio y se muestra la salida de la instalacion. Si Colab
solicita reiniciar la sesion, se reinicia y se continua desde la verificacion del
entorno, sin volver a clonar la carpeta existente.


In [ ]:
from pathlib import Path
import subprocess
import sys

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
resultado_instalacion = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(directorio_proyecto / "requirements_colab.txt")],
    capture_output=True,
    text=True,
)
print(resultado_instalacion.stdout, end="")
if resultado_instalacion.stderr:
    print(resultado_instalacion.stderr, end="")
resultado_instalacion.check_returncode()


## **Verificar la GPU y las versiones**

Se ejecuta despues de instalar las dependencias o reiniciar la sesion. Se registran
las versiones y el commit, y se comprueba que Numba detecte una GPU real. Esta
comprobacion todavia no demuestra que un kernel propio compile correctamente.


In [ ]:
from pathlib import Path
from importlib.metadata import version
import subprocess
import sys

from numba import cuda

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
commit_ejecutado = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=directorio_proyecto, text=True
).strip()
print("Python:", sys.version)
print("Commit:", commit_ejecutado)
for nombre_paquete in ("numpy", "numba", "numba-cuda"):
    print(nombre_paquete, version(nombre_paquete))

resultado_monitor_gpu = subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True
)
print(resultado_monitor_gpu.stdout, end="")
if resultado_monitor_gpu.stderr:
    print(resultado_monitor_gpu.stderr, end="")
resultado_monitor_gpu.check_returncode()

if cuda.config.ENABLE_CUDASIM:
    raise RuntimeError("El simulador no sirve para verificar la GPU real de Colab.")
if not cuda.is_available():
    raise RuntimeError("Numba no detecta CUDA. Revisa la GPU y las dependencias.")
cuda.detect()
print("GPU detectada. Continua con la prueba del kernel propio.")


## **Ejecutar un kernel CUDA propio**

Se ejecuta despues de detectar la GPU, para comprobar compilacion, transferencias
y calculo. El script suma 1000 elementos y compara el resultado con NumPy. Se
muestran tanto los resultados como las advertencias; esta prueba no mide rendimiento.


In [ ]:
from pathlib import Path
import subprocess
import sys

ruta_verificacion_cuda = Path("/content/Proyecto_3_Cuda_CNN/src/00_verificar_cuda.py")
resultado_ejecucion = subprocess.run(
    [sys.executable, str(ruta_verificacion_cuda)],
    capture_output=True,
    text=True,
)
print(resultado_ejecucion.stdout, end="")
if resultado_ejecucion.stderr:
    print(resultado_ejecucion.stderr, end="")
resultado_ejecucion.check_returncode()


## **Montar Drive y localizar los tensores**

Se ejecuta antes de cargar los datos y se autoriza la cuenta de cada integrante.
La carpeta `proyecto_senias` debe estar en Mi unidad o tener un acceso directo alli
con ese nombre. Si el archivo no aparece, se ajusta la ruta antes de continuar.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

ruta_tensores_drive = Path(
    "/content/drive/MyDrive/proyecto_senias/02_tensores.npz"
)
print("Archivo encontrado:", ruta_tensores_drive.is_file())
if not ruta_tensores_drive.is_file():
    raise FileNotFoundError(f"No se encontraron los tensores en {ruta_tensores_drive}")


## **Copiar y validar los datos**

Luego, se copia el archivo al disco temporal de Colab para poder trabajar sin
leer desde Drive durante el entrenamiento. `compartido/datos.py` verifica imagenes, etiquetas
y clases, conservando las particiones existentes de entrenamiento, validacion y prueba.


In [ ]:
from pathlib import Path
import shutil
import sys

directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
sys.path.insert(0, str(directorio_proyecto / "src"))

import config
from compartido.datos import cargar_datos

config.DIRECTORIO_DATOS_PROCESADOS.mkdir(parents=True, exist_ok=True)
shutil.copyfile(ruta_tensores_drive, config.RUTA_TENSORES)

datos_imagenes = cargar_datos()
for particion in ("train", "val", "test"):
    cantidad_imagenes = len(datos_imagenes[f"x_{particion}"])
    print(f"{particion}: {cantidad_imagenes} imagenes")


## **Entrenar la CNN en CPU**

Se ejecuta despues de copiar y validar los datos. Se comienza con una epoca y se
utiliza un nombre distinto por ejecucion para conservar los resultados anteriores.
El script separa calentamiento, entrenamiento y validacion, sin evaluar test.
La salida completa se muestra al terminar; esta corrida inicial no sustituye los
experimentos repetidos de comparacion CPU/GPU.


In [ ]:
from pathlib import Path
import subprocess
import sys

import config

nombre_ejecucion = "cpu_01"
cantidad_epocas = 1
tamano_lote = config.TAMANO_LOTE
directorio_proyecto = Path("/content/Proyecto_3_Cuda_CNN")
ruta_modelo_cpu = config.DIRECTORIO_MODELOS / f"{nombre_ejecucion}.npz"
ruta_registro_cpu = config.DIRECTORIO_RESULTADOS / f"{nombre_ejecucion}.json"

resultado_entrenamiento = subprocess.run(
    [
        sys.executable, str(directorio_proyecto / "src/entrenar_cpu.py"),
        "--epocas", str(cantidad_epocas),
        "--tamano-lote", str(tamano_lote),
        "--ruta-tensores", str(config.RUTA_TENSORES),
        "--ruta-modelo", str(ruta_modelo_cpu),
        "--ruta-registro", str(ruta_registro_cpu),
    ],
    cwd=directorio_proyecto,
    capture_output=True,
    text=True,
)
print(resultado_entrenamiento.stdout, end="")
if resultado_entrenamiento.stderr:
    print(resultado_entrenamiento.stderr, end="")
resultado_entrenamiento.check_returncode()


## **Respaldar los resultados en Drive**

Se ejecuta despues de que el entrenamiento termine correctamente y antes de cerrar
la sesion. Se copian los pesos y el registro a una carpeta con el nombre de la
corrida, dentro de `proyecto_senias/resultados`. Los pesos permiten predecir despues;
no incluyen el estado de Adam para reanudar el entrenamiento exactamente.


In [ ]:
import shutil

directorio_respaldo = ruta_tensores_drive.parent / "resultados" / nombre_ejecucion
if not ruta_modelo_cpu.is_file() or not ruta_registro_cpu.is_file():
    raise FileNotFoundError("Faltan los archivos de una ejecucion terminada.")
directorio_respaldo.mkdir(parents=True, exist_ok=False)

for ruta_resultado in (ruta_modelo_cpu, ruta_registro_cpu):
    ruta_respaldo = directorio_respaldo / ruta_resultado.name
    shutil.copyfile(ruta_resultado, ruta_respaldo)
    print("Respaldado:", ruta_respaldo)
